# 10m Wind Speed Analysis: Historical vs Control vs Projection

This notebook analyses the 10 metre wind speed over Scandinavia (including Finland) for **8 August 2023** (hourly data) across three storyline experiments and using 3 ensemble members:

| Label | Experiment key | Description |
|-------|---------------|-------------|
| Historical | `???` | Past/historical climate |
| Control | `???` | Present-day climate |
| Projection | `???` | +2 K warmer storyline |

**What we do:**
1. Download all ensemble members for all 24 hours of 8 Aug 2023 for each experiment
2. Compute ensemble-mean
3. Plot spatial wind speed *differences* between the experiments at 12 UTC

**References:**
- [Climate DT user guide](https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide)
- [ECMWF parameter DB – param 207](https://codes.ecmwf.int/grib/param-db/)
- John et al. 2026, *Global Kilometer-Scale Climate Storylines Using Spectral Nudging*

In [ ]:
import earthkit.data as ekd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import pandas as pd

## Configuration

**Tasks**:  Create a request for the required data. Use the variables below so you can easily modify the request. Regrid the data serverside to 0.1 degree. 
- `REALIZATIONS`: list of ensemble member numbers to request.
- `RESOLUTION`: `'standard'` (H128, ~50 km) is faster for testing; switch to `'high'` (H512, ~10 km) for production.

In [ ]:
EXPERIMENTS = []

REALIZATIONS = []    # ensemble members to attempt
DATE         = ''         # 8 August 2023
PARAM        = ''              # 10 metre wind speed
RESOLUTION   = ''             # 'standard' (~50 km) or 'high' (~10 km)
TIME         = ''             # single time step

# Server-side regridding target resolution
GRID = ''

# Scandinavia + Finland area: N/W/S/E
AREA    = '72/4/54/32'
ADDRESS = ''

## 1. Data Request and Download

We use **server-side regridding**: by adding `'grid'`, `'interpolation'`, and `'area'` keys to the request, Polytope reprojects the native HEALPix data to a regular lat/lon grid before sending it back. This means:
- The response is a standard GRIB file — `data.ls()` works normally.
- `to_xarray()` returns a dataset with proper `latitude` / `longitude` / `valid_time` dimensions instead of the unstructured `points` / `datetimes` dimensions from a bounding-box CovJSON response.
- All experiments land on **exactly the same grid**, so differences are computed by plain xarray subtraction — no interpolation step needed.

We loop over each experiment and ensemble member; failed realizations are caught and skipped.

**Task:** Fill in missing request fields.

In [ ]:
def build_request(experiment, realization):
    """Return a Polytope request dict for one experiment / realization."""
    return {
        'activity'      : '',
        'class'         : 'd1',
        'dataset'       : 'climate-dt',
        'experiment'    : experiment,
        'expver'        : '0001',
        'model'         : '',
        'generation'    : '2',
        'realization'   : realization,
        'resolution'    : RESOLUTION,
        'date'          : DATE,
        'time'          : TIME,
        'stream'        : '',
        'type'          : 'fc',
        'levtype'       : '',
        'param'         : PARAM,
        # Server-side regridding — returns regular lat/lon GRIB instead of HEALPix CovJSON
        'grid'          : GRID,
        'interpolation' : 'grid-box-average',
        'area'          : AREA,   # N/W/S/E  (replaces the bounding-box 'feature' key)
    }

In the next step we loop over all of the data requests and combine them into one dataset. An alternative is to make different datasets for each experiment type.

**TASK:** Fill in the blanks :)

In [ ]:
datasets = {}  # {exp: xr.Dataset with dim 'realization' prepended}

for exp in EXPERIMENTS:
    print(f'\n=== Experiment: {exp} ===')
    realization_datasets = []

    for r in REALIZATIONS:
        print(f'  Requesting realization {r} ...', end=' ', flush=True)
        
        # TODO: Get data with earthkit
        data = 

        # TODO: Convert to xarray
        ds = 

        # Tag with the realization number so we can concatenate cleanly
        # TODO collectively

    datasets[exp] = xr.concat(realization_datasets, dim='realization')
    print(f'  Dataset shape: {dict(datasets[exp].dims)}')

print('\nDownload complete.')

## 2. Data Inspection

Let's check the structure of the downloaded data and identify the wind speed variable name.

In [ ]:
# Show the first experiment's dataset as a representative example
first_exp = EXPERIMENTS[0]
print(f'Structure of {first_exp} dataset:')
print(datasets[first_exp])

print('\nData variables available:')
for exp in datasets:
    print(f'  {exp}: {list(datasets[exp].data_vars)}')

## 3. Spatial Wind Speed Differences at 12 UTC

Because we used server-side regridding, every experiment is already on the same 0.1° lat/lon grid. Differences are computed by plain xarray subtraction — no interpolation needed.

We plot three panels:
- **Projection − Historical** (Tplus2.0K − hist)
- **Projection − Control** (Tplus2.0K − cont)
- **Control − Historical** (cont − hist)

In [ ]:
# Ensemble-mean spatial field for each experiment (single time step → squeeze time dim)
# Result is a DataArray with dims (latitude, longitude)
# TODO collectively

# Pairwise differences — direct xarray arithmetic, same grid guaranteed
# TODO collectively

In [ ]:
# Symmetric colour scale: use 98th-percentile absolute value across all three differences
all_vals = np.concatenate([d.values[~np.isnan(d.values)].ravel() for d in diffs.values()])
vmax = float(np.nanpercentile(np.abs(all_vals), 98))
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# lat/lon come straight from the xarray coordinates — no linspace needed
lat_out = 
lon_out = 

# Multiple subplots are required
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

# TODO: Loop over the data sets to get a subplot showing the difference between the experiments.

fig.suptitle(
    f'10m Wind Speed Differences at {TIME} UTC — {DATE}\n'
    f'Ensemble mean, Scandinavia + Finland, {GRID}° grid',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

## Summary

| Panel | What it shows |
|-------|---------------|
| **Tplus2.0K − hist** | Difference between historical forcings and the +2 K storyline warming |
| **Tplus2.0K − cont** | Wind response in the +2 K future vs the 1950s |
| **cont − hist** | Difference between past and present |

Red = stronger wind in the first experiment; blue = weaker. The symmetric diverging scale is set to the 98th-percentile absolute difference to avoid outlier saturation.